# Qwen3-14B 베이스 모델 → Google Drive 업로드

배포되는 신용평가 시스템(GCP Cloud Run)이 **HuggingFace 429를 우회**하도록,
베이스 모델(Qwen3-14B, ~28GB)을 어댑터와 같은 방식으로 Drive 에 올린다.

- Colab 은 HF 접근이 원활하므로 여기서 받아 Drive 로 복사한다.
- Cloud Run 은 이 Drive 폴더를 서비스 계정(ADC)으로 읽는다(HF 안 거침).

**런타임 유형은 GPU 불필요** (다운로드/복사만). CPU 런타임으로 충분.

**소요**: HF 다운로드 + Drive 복사로 약 20~40분 (네트워크에 따라).

## 1. Drive 마운트 + 대상 폴더 생성

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# 어댑터(Qwen3_fintech) 옆에 베이스 모델 폴더를 둔다.
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/Qwen3_base'
os.makedirs(BASE_DIR, exist_ok=True)
print('베이스 모델 저장 폴더:', BASE_DIR)

## 2. HuggingFace 에서 Qwen3-14B 다운로드
추론에 필요한 파일만 받는다(safetensors 가중치 + 설정 + 토크나이저).

In [ ]:
!pip install -q -U huggingface_hub
from huggingface_hub import snapshot_download

local = snapshot_download(
    'Qwen/Qwen3-14B',
    allow_patterns=['*.json', '*.txt', '*.safetensors', 'merges.txt', 'tokenizer*', '*.jinja'],
)
print('HF 캐시 경로:', local)
!ls -lh "$local" | head -30

## 3. Drive 로 복사 (평면 폴더 — 하위 폴더 없이 파일만)
Cloud Run 다운로더는 폴더 바로 아래의 파일만 받으므로, 평면으로 복사한다.

In [ ]:
import shutil, os, glob
# snapshot_download 는 심볼릭 링크를 쓰므로 실제 파일을 따라가 복사
count = 0
for f in glob.glob(os.path.join(local, '*')):
    if os.path.isfile(f) or os.path.islink(f):
        real = os.path.realpath(f)
        dest = os.path.join(BASE_DIR, os.path.basename(f))
        if os.path.exists(dest) and os.path.getsize(dest) == os.path.getsize(real):
            print('건너뜀(이미 있음):', os.path.basename(f)); count += 1; continue
        print('복사:', os.path.basename(f), f'({os.path.getsize(real)/1e6:.1f} MB)')
        shutil.copy2(real, dest)
        count += 1
print(f'\n복사 완료: {count} 개 파일 → {BASE_DIR}')

## 4. 검증 + 폴더 ID 확인
config.json 과 8개 safetensors 가 있어야 한다. 출력된 폴더 ID 를 배포에 사용.

In [ ]:
import os, glob
files = sorted(os.path.basename(f) for f in glob.glob(os.path.join(BASE_DIR, '*')))
print('파일 수:', len(files))
for f in files: print('  ', f)
assert 'config.json' in files, 'config.json 누락!'
st = [f for f in files if f.endswith('.safetensors')]
print(f'\nsafetensors: {len(st)} 개 (8개 기대)')

# 폴더 ID 조회
try:
    from googleapiclient.discovery import build
    from google.colab import auth
    auth.authenticate_user()
    svc = build('drive', 'v3')
    q = "name='Qwen3_base' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    res = svc.files().list(q=q, fields='files(id,name)').execute().get('files', [])
    if res:
        print('\nQwen3_base 폴더 ID:', res[0]['id'])
        print('→ 배포 서버에서 CSS_LLM_BASE_DRIVE_FOLDER_ID 로 사용하세요.')
except Exception as e:
    print('자동 조회 실패:', e, '— Drive URL 의 /folders/<ID> 를 사용하세요.')

## 5. 서비스 계정에 폴더 공유 (중요)
Cloud Run 런타임 서비스 계정이 이 폴더를 읽을 수 있어야 한다.

Drive 웹에서 **Qwen3_base 폴더 우클릭 → 공유** →
아래 이메일을 **뷰어**로 추가 (알림 보내기 해제):

```
css-rating2-run@qwen3-fintech.iam.gserviceaccount.com
```